## 🧭 클론해서 뭘 하면 되나요? (3단계)

1. **오른쪽 설정(Settings)** → **Accelerator: GPU** 선택, **반드시 Internet: On** (HuggingFace에서 모델 다운로드함).
2. **아래 "1. 리포 준비" 셀**에서 `REPO_URL` **한 줄만** 본인 GitHub AIMO 주소로 바꾼 뒤 **그 셀 실행** (예: `https://github.com/내아이디/AIMO.git`).
3. **나머지 셀을 위에서부터 순서대로 실행** (또는 메뉴에서 Run → Run All).
   - 2번째 셀(대안)은 **클론했으면 건너뛰기**.
   - "2. 패키지 설치" → **(없으면) "2.5 데이터 준비"** → "3. Numina 평가 실행" 순서로 실행하면 끝.

끝나면 오른쪽 **Output**에서 `results` 폴더를 다운로드하면 됩니다.  
**전체 과정·검증 포인트:** 리포의 `docs/run-eval/KAGGLE_SETUP_PROCESS.md` 참고.

# Numina 평가 — Kaggle에서 실행

**설정:** 오른쪽 설정(Settings)에서 **Accelerator → GPU** + **Internet: On** 선택 후 실행하세요. (Internet 꺼져 있으면 HuggingFace 모델 다운로드 불가.)

- 이 노트북은 AIMO 리포지토리를 클론한 뒤 Numina 60문항 평가를 실행합니다.
- 결과는 `/kaggle/working/results/`에 저장됩니다. 제출 후 Output에서 다운로드할 수 있습니다.
- **소요 시간:** 1.5B 기준 약 20~40분 (문항당 타임아웃 15분 적용).

## 1. 리포지토리 준비

**👇 아래 코드 셀에서 `REPO_URL` 한 줄만 본인 GitHub AIMO 주소로 바꾸고 실행하세요.**  
(예: GitHub에서 AIMO 리포 열고 → 초록색 Code 버튼 → HTTPS 주소 복사 → `.git` 붙이기 → `REPO_URL = "https://github.com/내아이디/AIMO.git"`)

In [ ]:
# 👇 클론할 리포 주소 (이미 lilyth-y/AIMO 로 설정됨)
REPO_URL = "https://github.com/lilyth-y/AIMO.git"
WORK_DIR = "/kaggle/working/AIMO"
BRANCH = "changes"  # 푸시한 브랜치와 맞추기

import os
from pathlib import Path

if Path(WORK_DIR).exists():
    print(f"[OK] 이미 존재: {WORK_DIR} — GitHub 최신 변경사항 반영을 위해 pull")
    !git -C {WORK_DIR} checkout {BRANCH}
    !git -C {WORK_DIR} pull origin {BRANCH}
    print("[OK] pull 완료 (변경사항 적용됨)")
else:
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {WORK_DIR}
    print(f"[OK] 클론 완료: {WORK_DIR}")
os.chdir(WORK_DIR)
print("CWD:", os.getcwd())

In [ ]:
# (대안) Kaggle Dataset으로 AIMO를 추가한 경우: 입력 경로를 복사해서 working에서 실행
# 위에서 클론했다면 이 셀은 건너뛰세요.

# INPUT_AIMO = "/kaggle/input/aimo"  # Dataset 이름에 맞게 수정
# WORK_DIR = "/kaggle/working/AIMO"
# import shutil
# if Path(INPUT_AIMO).exists() and not Path(WORK_DIR).exists():
#     shutil.copytree(INPUT_AIMO, WORK_DIR)
#     os.chdir(WORK_DIR)
#     print("CWD:", os.getcwd())
# else:
#     print("이미 WORK_DIR 있거나 INPUT_AIMO 없음. 위 클론 셀 사용.")

## 2. 패키지 설치

클론이 끝났으면 **이 셀 실행** → 필요한 패키지 설치 (1~2분 정도).

In [ ]:
# 클론된 리포 안에 있으므로 경로를 지정해서 설치 (Kaggle 셀은 기본 CWD가 /kaggle/working 이라서)
!pip install -q -r /kaggle/working/AIMO/requirements-kaggle.txt
print("설치 완료.")

## 2.5 데이터 준비 (numina_eval_balanced.json)

리포에 평가 데이터 파일이 없으면 **이 셀 실행** 시 HuggingFace에서 NuminaMath-1.5를 받아 60문항 세트를 만듭니다 (최초 1회, 수 분 소요). 이미 파일이 있으면 건너뛰세요.

In [ ]:
WORK_DIR = "/kaggle/working/AIMO"
DATA_FILE = f"{WORK_DIR}/data/numina_eval_balanced.json"

if __import__("pathlib").Path(DATA_FILE).exists():
    print("[OK] numina_eval_balanced.json 이미 있음. 건너뜀.")
else:
    import os, sys
    os.makedirs(f"{WORK_DIR}/data", exist_ok=True)
    sys.path.insert(0, f"{WORK_DIR}/src")
    os.chdir(WORK_DIR)
    from data.numina_loader import NuminaMathDataLoader
    loader = NuminaMathDataLoader(version="1.5", cache_dir=f"{WORK_DIR}/data/numina_cache")
    loader.load_dataset(streaming=True)
    loader.create_evaluation_set(
        n_easy=10, n_medium=20, n_hard=30,
        output_file=DATA_FILE
    )
    print("[OK] numina_eval_balanced.json 생성 완료.")

## 3. Numina 평가 실행 (모델 돌리기)

**이 셀 실행하면 실제로 모델이 돌아갑니다.** 60문항 기준 20~40분 걸릴 수 있어요.  

- **"outgoing traffic has been disabled"** 가 나오면: Kaggle이 **외부 네트워크를 막은 상태**입니다.  
  → 오른쪽 **Settings** → **Internet** 항목을 **On** 으로 바꾼 뒤 **Save** (또는 Save Version) 한 다음, 이 셀을 다시 실행하세요.  
  (일부 노트북은 전화번호 인증이 있어야 Internet을 켤 수 있습니다. 대회 제출용 노트북은 Internet을 켤 수 없을 수 있어, 그 경우 모델을 Dataset으로 미리 올려 두어야 합니다.)  
- 5문항만 빠르게 테스트하려면 아래 코드에서 `# os.environ["MAX_PROBLEMS"] = "5"` 주석을 해제한 뒤 실행하세요.

In [ ]:
import os
import sys
import subprocess

WORK_DIR = "/kaggle/working/AIMO"

# (선택) Kaggle Dataset으로 받은 로컬 모델 경로가 있으면 여기에 넣기 (Internet 없이 실행 가능)
KAGGLE_MODEL_PATH = "/kaggle/input/models/serhiikharchuk/qwen2-math-1.5b-instruct/transformers/qwen2-math-1.5b-instruct-bf16/1"
# (선택) 5문항만 빠르게 테스트하려면 아래 주석 해제
# os.environ["MAX_PROBLEMS"] = "5"

# Kaggle 환경 설정
if KAGGLE_MODEL_PATH and os.path.exists(KAGGLE_MODEL_PATH):
    os.environ["OMI_MODEL"] = KAGGLE_MODEL_PATH
    print("[Kaggle] Using local model:", KAGGLE_MODEL_PATH)
else:
    os.environ["OMI_MODEL"] = "Qwen/Qwen2.5-Math-1.5B-Instruct"
    print("[Kaggle] Using HuggingFace repo (Internet 필요)")
os.environ.setdefault("AIMO_RESULTS_DIR", "/kaggle/working")
os.environ.setdefault("EVAL_PROBLEM_TIMEOUT", "900")
if not os.environ.get("AIMO_DATA_DIR"):
    if os.path.exists(f"{WORK_DIR}/data/numina_eval_balanced.json"):
        os.environ["AIMO_DATA_DIR"] = f"{WORK_DIR}/data"
print("[Kaggle] OMI_MODEL =", os.environ.get("OMI_MODEL"))
print("[Kaggle] AIMO_RESULTS_DIR =", os.environ.get("AIMO_RESULTS_DIR"))
print("[Kaggle] AIMO_DATA_DIR =", os.environ.get("AIMO_DATA_DIR", "(auto)"))

child_env = {
    **os.environ,
    "OMI_MODEL": os.environ["OMI_MODEL"],
    "HF_HUB_OFFLINE": "0",
    "TRANSFORMERS_OFFLINE": "0",
}

# examples/run_numina_evaluation.py 직접 실행 (scripts/ 파일 불필요)
rc = subprocess.run(
    [sys.executable, f"{WORK_DIR}/examples/run_numina_evaluation.py"],
    cwd=WORK_DIR,
    env=child_env,
    stdin=subprocess.DEVNULL,
)
if rc.returncode != 0:
    raise SystemExit(rc.returncode)

## 4. 결과 확인

결과는 `/kaggle/working/results/`에 저장됩니다. 노트북 제출 후 **Output** 탭에서 `results` 폴더를 다운로드하세요.

In [ ]:
!ls -la /kaggle/working/results/ 2>/dev/null || echo "results 폴더가 아직 없거나 경로가 다릅니다."